# Testes de hipótese: infraestrutura escolar e IDEB

Este notebook parte da base integrada gerada por `01_exploracao_dados.ipynb`. Execute primeiro o notebook 01 para criar `Dados/Tratados/painel_escolas_sp_2023_2025_com_ideb.csv`.

A unidade de análise é a escola pública de São Paulo presente nos Censos Escolares de 2023 e 2025, com IDEB numérico nos dois anos.


## Perguntas e hipóteses

Para cada recurso, a pergunta é: entre escolas que não o tinham em 2023, as que passaram a tê-lo em 2025 apresentaram maior variação do IDEB do que as que continuaram sem ele? A variação é `ΔIDEB = IDEB_2025 − IDEB_2023`.

Serão analisados quatro indicadores: internet para aprendizagem, banda larga, laboratório de informática e biblioteca. Em cada comparação, o grupo de interesse é **0→1** e o grupo de referência é **0→0**. As transições **1→0** e **1→1** não entram nessa comparação porque as escolas já tinham o recurso na linha de base. Linhas sem IDEB ou sem classificação válida do indicador também são excluídas; o notebook informa as contagens.

- **H₀:** a distribuição de `ΔIDEB` no grupo 0→1 não tende a ser maior que no grupo 0→0.
- **H₁:** a distribuição de `ΔIDEB` no grupo 0→1 tende a ser maior que no grupo 0→0.

Cada comparação usa Mann–Whitney unilateral, com postos médios, correção para empates e continuidade, e aproximação normal para o p-valor. O ajuste de Holm considera a família dos quatro testes e controla o erro familiar. A internet para aprendizagem permanece como hipótese principal; as outras três são exploratórias. Os resultados indicam associação, não causalidade.


In [1]:
from pathlib import Path
from math import sqrt
from statistics import NormalDist

import numpy as np
import pandas as pd

# Resolve a raiz do projeto tanto ao abrir o notebook na raiz quanto em Notebooks/.
PASTA_PROJETO = (
    Path("..") if Path.cwd().name.lower() in {"notebooks", "notebook"}
    else Path(".")
)
caminho_base = (
    PASTA_PROJETO / "Dados" / "Tratados" /
    "painel_escolas_sp_2023_2025_com_ideb.csv"
)
if not caminho_base.exists():
    raise FileNotFoundError(
        "Execute primeiro 01_exploracao_dados.ipynb para gerar a base integrada."
    )

indicadores = {
    "Internet para aprendizagem": "IN_INTERNET_APRENDIZAGEM",
    "Banda larga": "IN_BANDA_LARGA",
    "Laboratório de informática": "IN_LABORATORIO_INFORMATICA",
    "Biblioteca": "IN_BIBLIOTECA",
}
colunas_ideb = ["VL_OBSERVADO_2023", "VL_OBSERVADO_2025"]
colunas_indicadores = [
    f"{indicador}_{ano}"
    for indicador in indicadores.values()
    for ano in (2023, 2025)
]
colunas = colunas_indicadores + colunas_ideb

print("Lendo somente as colunas dos quatro indicadores e do IDEB...", flush=True)
df = pd.read_csv(caminho_base, usecols=colunas, encoding="utf-8-sig")
print(f"Linhas carregadas: {len(df):,}.", flush=True)
for coluna in colunas:
    df[coluna] = pd.to_numeric(df[coluna], errors="coerce")

# IDEB precisa estar disponível nos dois anos para calcular a variação.
df_ideb = df.dropna(subset=colunas_ideb).copy()
df_ideb["DELTA_IDEB"] = (
    df_ideb["VL_OBSERVADO_2025"] - df_ideb["VL_OBSERVADO_2023"]
)
print(f"Escolas com IDEB nos dois anos: {len(df_ideb):,}.", flush=True)

resultados = []
for nome, indicador in indicadores.items():
    flag_2023 = f"{indicador}_2023"
    flag_2025 = f"{indicador}_2025"
    dados = df_ideb.dropna(subset=[flag_2023, flag_2025]).copy()

    if not dados[[flag_2023, flag_2025]].isin([0, 1]).all().all():
        raise ValueError(f"{indicador} precisa conter somente 0, 1 ou ausente.")

    grupo_mudou = dados.loc[
        dados[flag_2023].eq(0) & dados[flag_2025].eq(1), "DELTA_IDEB"
    ].to_numpy(dtype=float)
    grupo_estavel_sem = dados.loc[
        dados[flag_2023].eq(0) & dados[flag_2025].eq(0), "DELTA_IDEB"
    ].to_numpy(dtype=float)
    outras_transicoes = int(
        (dados[flag_2023].eq(1)).sum()
    )

    n_mudou = len(grupo_mudou)
    n_estavel_sem = len(grupo_estavel_sem)
    if n_mudou == 0 or n_estavel_sem == 0:
        raise ValueError(f"{nome}: os grupos 0→1 e 0→0 precisam ter escolas.")

    # Mann–Whitney: U e efeito rank-biserial, com o grupo 0→1 primeiro.
    combinados = np.concatenate([grupo_mudou, grupo_estavel_sem])
    postos = pd.Series(combinados).rank(method="average").to_numpy()
    u = float(postos[:n_mudou].sum() - n_mudou * (n_mudou + 1) / 2)
    contagens_empates = pd.Series(combinados).value_counts().to_numpy()
    termo_empates = sum(int(n) ** 3 - int(n) for n in contagens_empates)
    n_total = n_mudou + n_estavel_sem
    variancia_u = (
        n_mudou * n_estavel_sem / 12 *
        (n_total + 1 - termo_empates / (n_total * (n_total - 1)))
    )
    if variancia_u <= 0:
        raise ValueError(f"{nome}: não foi possível calcular a variância de U.")

    z = (u - n_mudou * n_estavel_sem / 2 - 0.5) / sqrt(variancia_u)
    p_bruto = 1 - NormalDist().cdf(z)
    efeito = 2 * u / (n_mudou * n_estavel_sem) - 1
    n_ausente = len(df_ideb) - len(dados)

    resultados.append({
        "Indicador": nome,
        "n_0→1": n_mudou,
        "n_0→0": n_estavel_sem,
        "n_outras_transições": outras_transicoes,
        "n_indicador_ausente": n_ausente,
        "média_ΔIDEB_0→1": float(grupo_mudou.mean()),
        "média_ΔIDEB_0→0": float(grupo_estavel_sem.mean()),
        "mediana_ΔIDEB_0→1": float(np.median(grupo_mudou)),
        "mediana_ΔIDEB_0→0": float(np.median(grupo_estavel_sem)),
        "U": u,
        "p_bruto": p_bruto,
        "efeito_rank_biserial": efeito,
    })

# Ajuste step-down de Holm aplicado conjuntamente aos quatro p-valores.
ordem = sorted(range(len(resultados)), key=lambda i: resultados[i]["p_bruto"])
p_ajustado_anterior = 0.0
quantidade_testes = len(resultados)
for posicao, indice in enumerate(ordem):
    fator = quantidade_testes - posicao
    p_ajustado = min(1.0, fator * resultados[indice]["p_bruto"])
    p_ajustado_anterior = max(p_ajustado_anterior, p_ajustado)
    resultados[indice]["p_Holm"] = p_ajustado_anterior

resumo = pd.DataFrame(resultados)
print("\nResultados (α = 0,05; p-Holm ajustado para quatro testes):")
for resultado in resultados:
    decisao = (
        "evidência a favor de H₁ após Holm"
        if resultado["p_Holm"] < 0.05
        else "H₁ não confirmada após Holm"
    )
    print(
        f"\n{resultado['Indicador']} — 0→1: {resultado['n_0→1']}; "
        f"0→0: {resultado['n_0→0']}; "
        f"outras transições: {resultado['n_outras_transições']}; "
        f"indicador ausente: {resultado['n_indicador_ausente']}"
    )
    print(
        f"  média ΔIDEB: 0→1 = {resultado['média_ΔIDEB_0→1']:.3f}; "
        f"0→0 = {resultado['média_ΔIDEB_0→0']:.3f}"
    )
    print(
        f"  mediana ΔIDEB: 0→1 = {resultado['mediana_ΔIDEB_0→1']:.3f}; "
        f"0→0 = {resultado['mediana_ΔIDEB_0→0']:.3f}"
    )
    print(
        f"  U = {resultado['U']:.1f}; p bruto = {resultado['p_bruto']:.4f}; "
        f"p Holm = {resultado['p_Holm']:.4f}; "
        f"efeito rank-biserial = {resultado['efeito_rank_biserial']:.4f}"
    )
    print(f"  Decisão: {decisao}.")


Lendo somente as colunas dos quatro indicadores e do IDEB...
Linhas carregadas: 5,445.
Escolas com IDEB nos dois anos: 4,437.

Resultados (α = 0,05; p-Holm ajustado para quatro testes):

Internet para aprendizagem — 0→1: 67; 0→0: 27; outras transições: 4343; indicador ausente: 0
  média ΔIDEB: 0→1 = 0.064; 0→0 = 0.015
  mediana ΔIDEB: 0→1 = 0.000; 0→0 = 0.100
  U = 913.0; p bruto = 0.4733; p Holm = 0.9466; efeito rank-biserial = 0.0094
  Decisão: H₁ não confirmada após Holm.

Banda larga — 0→1: 65; 0→0: 118; outras transições: 4248; indicador ausente: 6
  média ΔIDEB: 0→1 = 0.100; 0→0 = 0.074
  mediana ΔIDEB: 0→1 = 0.100; 0→0 = 0.100
  U = 4026.5; p bruto = 0.2884; p Holm = 0.8653; efeito rank-biserial = 0.0499
  Decisão: H₁ não confirmada após Holm.

Laboratório de informática — 0→1: 146; 0→0: 670; outras transições: 3621; indicador ausente: 0
  média ΔIDEB: 0→1 = 0.097; 0→0 = 0.050
  mediana ΔIDEB: 0→1 = 0.100; 0→0 = 0.100
  U = 51468.0; p bruto = 0.1604; p Holm = 0.6417; efeito rank